# Data Cleaning & Structural Validation
**Workora Task:** Clean a messy dataset by removing duplicates, handling missing values and validating data types.

**Workflow:** Load → Inspect → Fix headers → Fix data types → Standardize text → Duplicates → Missing values → Outliers → Validate → Export.
Every decision is recorded in a `cleaning_log` and exported as `cleaning_log.csv`.

In [ ]:
import pandas as pd, numpy as np, re, warnings
warnings.filterwarnings("ignore")

RAW_PATH   = "raw_dataset.csv"      # <-- put the downloaded Workora file name here
CLEAN_PATH = "cleaned_dataset.csv"
log = []
def record(step, action, detail, rows_affected=""):
    log.append({"step": step, "action": action, "detail": detail, "rows_affected": rows_affected})
    print(f"[{step}] {action}: {detail} {('('+str(rows_affected)+')') if rows_affected!='' else ''}")

## 1. Load & Inspect

In [ ]:
df = pd.read_csv(RAW_PATH, encoding="utf-8", encoding_errors="replace")
raw_shape = df.shape
print("Shape:", raw_shape)
display(df.head())
df.info()

In [ ]:
print("Missing values per column:"); display(df.isna().sum().to_frame("missing").assign(pct=lambda x:(x.missing/len(df)*100).round(1)))
print("Exact duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

## 2. Standardize column headers
Lowercase, trim, replace spaces/symbols with underscores.

In [ ]:
def clean_col(c):
    c = str(c).strip().lower()
    c = re.sub(r"[^0-9a-z]+", "_", c)
    return c.strip("_")
old = list(df.columns)
df.columns = [clean_col(c) for c in df.columns]
record("headers","standardized",f"{old} -> {list(df.columns)}")

## 3. Trim whitespace & normalize placeholder nulls

In [ ]:
NULL_TOKENS = {"", "na", "n/a", "nan", "null", "none", "-", "--", "?", "unknown", "missing"}
obj_cols = df.select_dtypes(include="object").columns
for c in obj_cols:
    s = df[c].astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    mask = s.str.lower().isin(NULL_TOKENS)
    df[c] = s.mask(mask, pd.NA)
    if mask.sum(): record("nulls","placeholder->NaN",f"'{c}' had placeholder nulls", int(mask.sum()))
record("text","trimmed whitespace",f"{len(obj_cols)} text columns")

## 4. Fix data types
Auto-detect: currency/number strings → numeric, date-like columns → datetime.

In [ ]:
def try_numeric(s):
    cleaned = s.astype("string").str.replace(r"[,\$€£₹%\s]", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

def try_date(s):
    # parse each value individually so mixed formats work
    return s.apply(lambda v: pd.to_datetime(v, errors="coerce", dayfirst=True) if pd.notna(v) else pd.NaT)

for c in df.select_dtypes(include=["object","string"]).columns:
    nonnull = df[c].dropna()
    if nonnull.empty: continue
    num = try_numeric(df[c])
    if num.notna().sum() / len(nonnull) >= 0.9:
        df[c] = num; record("dtype","->numeric",f"'{c}'"); continue
    if re.search(r"date|time|dob|joined|created", c):
        d = try_date(df[c])
        if d.notna().sum() / len(nonnull) >= 0.8:
            df[c] = pd.to_datetime(d); record("dtype","->datetime",f"'{c}'"); continue
df.dtypes.to_frame("dtype")

## 5. Standardize categorical strings
Consistent casing, and a mapping for known variants (edit `VARIANT_MAP` for your dataset).

In [ ]:
VARIANT_MAP = {"m":"Male","male":"Male","f":"Female","female":"Female"}
for c in df.select_dtypes(include=["object","string"]).columns:
    before = df[c].nunique()
    if "name" in c: df[c] = df[c].str.title()
    else:           df[c] = df[c].str.title()
    low = df[c].str.lower()
    if set(low.dropna().unique()) & set(VARIANT_MAP) and low.nunique() <= 8:
        df[c] = low.map(VARIANT_MAP).fillna(df[c])
    record("categorical","standardized",f"'{c}': {before} -> {df[c].nunique()} unique values")
    print(c, sorted(df[c].dropna().unique())[:15])

## 6. Remove duplicates

In [ ]:
n = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
record("duplicates","dropped exact duplicate rows","kept first occurrence", int(n))

# ID-like column duplicates (same ID, different content)?
id_cols = [c for c in df.columns if c.endswith("_id") or c == "id"]
for c in id_cols:
    k = df[c].duplicated().sum()
    if k:
        df = df.drop_duplicates(subset=c, keep="first").reset_index(drop=True)
        record("duplicates","dropped duplicate key",f"'{c}' kept first", int(k))

## 7. Handle missing values
**Documented reasoning**
- Rows missing > 50 % of their fields carry little information → **drop**.
- ID columns can never be imputed → **drop** rows with missing ID.
- Numeric: **median** (robust to outliers). Categorical: **mode** if the column is < 5 % missing, otherwise the label `Unknown`. Dates: left as `NaT` (inventing dates would bias time analysis) — flagged instead.

In [ ]:
thr = len(df.columns) * 0.5
sparse = df.isna().sum(axis=1) > thr
if sparse.sum(): record("missing","dropped sparse rows",">50% fields missing", int(sparse.sum()))
df = df[~sparse].copy()

for c in id_cols:
    m = df[c].isna().sum()
    if m: df = df[df[c].notna()]; record("missing","dropped rows","missing ID in "+c, int(m))

for c in df.columns:
    m = df[c].isna().sum()
    if m == 0: continue
    if pd.api.types.is_numeric_dtype(df[c]):
        med = df[c].median(); df[c] = df[c].fillna(med)
        record("missing","imputed median",f"'{c}' = {med:.2f}", int(m))
    elif pd.api.types.is_datetime64_any_dtype(df[c]):
        df[c+"_missing_flag"] = df[c].isna()
        record("missing","left as NaT + flag column",f"'{c}'", int(m))
    else:
        if m/len(df) < 0.05: fill = df[c].mode().iloc[0]; how="mode"
        else: fill="Unknown"; how="'Unknown' label"
        df[c] = df[c].fillna(fill); record("missing",f"imputed {how}",f"'{c}' -> {fill}", int(m))
print("Remaining nulls (excluding intentional NaT):"); print(df.drop(columns=[c for c in df if pd.api.types.is_datetime64_any_dtype(df[c])]).isna().sum().sum())

## 8. Outliers (IQR rule)
Values outside Q1−1.5·IQR … Q3+1.5·IQR are inspected. Impossible values (e.g. negative age, age > 120) are treated as errors and winsorized (capped) rather than deleted, preserving the row.

In [ ]:
num_cols = [c for c in df.select_dtypes(include="number").columns if c not in id_cols]
for c in num_cols:
    q1,q3 = df[c].quantile([.25,.75]); iqr = q3-q1
    lo,hi = q1-1.5*iqr, q3+1.5*iqr
    if df[c].min() >= 0: lo = max(lo, 0)  # non-negative column
    out = (df[c]<lo)|(df[c]>hi)
    if out.sum():
        df[c+"_outlier_flag"] = out
        df[c] = df[c].clip(lo, hi)
        record("outliers","capped (winsorized) + flagged",f"'{c}' to [{lo:.1f}, {hi:.1f}]", int(out.sum()))
# domain rules
for c in num_cols:
    if "age" in c:
        bad = (df[c]<0)|(df[c]>120)
        if bad.sum(): df.loc[bad,c]=df[c].median(); record("validation","invalid age -> median",c,int(bad.sum()))

## 9. Validate

In [ ]:
checks = {
 "no exact duplicate rows": df.duplicated().sum()==0,
 "no nulls in non-date columns": df.drop(columns=[c for c in df if pd.api.types.is_datetime64_any_dtype(df[c])]).isna().sum().sum()==0,
 "headers are snake_case": all(re.fullmatch(r"[a-z0-9_]+", c) for c in df.columns),
 "no leading/trailing spaces": all((df[c].dropna().astype(str)==df[c].dropna().astype(str).str.strip()).all() for c in df.select_dtypes(include=["object","string"]).columns),
}
for k,v in checks.items(): print("PASS" if v else "FAIL", "-", k)
print("\nRaw shape:", raw_shape, "-> Clean shape:", df.shape)
df.dtypes.to_frame("dtype")

## 10. Export

In [ ]:
df.to_csv(CLEAN_PATH, index=False)
pd.DataFrame(log).to_csv("cleaning_log.csv", index=False)
try:
    with pd.ExcelWriter("cleaned_dataset.xlsx") as xw:
        df.to_excel(xw, sheet_name="Cleaned Data", index=False)
        pd.DataFrame(log).to_excel(xw, sheet_name="Cleaning Log", index=False)
except Exception as e: print("Excel export skipped:", e)
display(pd.DataFrame(log))
display(df.head(10))

## Summary of approach
| Issue | Decision | Why |
|---|---|---|
| Messy headers | snake_case | Safe for code & SQL |
| Whitespace / casing variants | trim + title-case + mapping | Merges "chennai ", "CHENNAI" |
| Currency strings | strip symbols → numeric | Enables math |
| Mixed date formats | parse individually | Consistent datetime |
| Duplicates | drop exact + duplicate keys | Prevents double counting |
| Missing numeric | median | Robust to outliers |
| Missing categorical | mode / "Unknown" | Keeps rows |
| Outliers | IQR cap + flag | Preserves data, limits distortion |